# Configuración del entorno e instalación de dependencias
En este primer bloque se instalan y cargan las librerías necesarias para el procesamiento, descarga externa de datos y análisis exploratorio.

In [1]:
!pip install -q gdown pandas numpy matplotlib seaborn plotly

import os
import gdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("Entorno configurado correctamente.")

Entorno configurado correctamente.


# Descarga automatizada de fuentes de datos desde Google Drive
Para garantizar la reproducibilidad independiente del entorno, se descargan los archivos CSV utilizando los enlaces públicos de Google Drive mediante gdown.

In [2]:
# Reemplazar 'ID_DEL_ARCHIVO' con el ID correspondiente extraído del enlace de compartir de cada archivo
archivos_drive = {
    'BaseUnificadaEstaciones.csv': '1I83lxYClCRQQThE-jyvxqwMPyVg9gpmW',
    'registro-historico-del-precio-del-boleto.csv': '1Db3YE6woTZcpWJEWQ8uWc6JdFpBoujOx',
    'viajes_anual.csv': '17hkOlBQq24sZzdTBRPdEo0r2ZHFx7NYx'
}

for nombre_archivo, file_id in archivos_drive.items():
    if not os.path.exists(nombre_archivo):
        url = f'https://drive.google.com/uc?id={file_id}'
        print(f"Descargando {nombre_archivo}...")
        gdown.download(url, nombre_archivo, quiet=False)
    else:
        print(f"{nombre_archivo} ya se encuentra descargado.")

Descargando BaseUnificadaEstaciones.csv...


Downloading...
From: https://drive.google.com/uc?id=1I83lxYClCRQQThE-jyvxqwMPyVg9gpmW
To: /content/BaseUnificadaEstaciones.csv
100%|██████████| 30.5M/30.5M [00:00<00:00, 66.5MB/s]


Descargando registro-historico-del-precio-del-boleto.csv...


Downloading...
From: https://drive.google.com/uc?id=1Db3YE6woTZcpWJEWQ8uWc6JdFpBoujOx
To: /content/registro-historico-del-precio-del-boleto.csv
100%|██████████| 6.04k/6.04k [00:00<00:00, 13.2MB/s]


Descargando viajes_anual.csv...


Downloading...
From: https://drive.google.com/uc?id=17hkOlBQq24sZzdTBRPdEo0r2ZHFx7NYx
To: /content/viajes_anual.csv
100%|██████████| 828/828 [00:00<00:00, 3.31MB/s]


# Carga preliminar, muestreo y dimensiones de los datasets
Se cargan los archivos tabulares en estructuras de datos de Pandas y se efectúa una inspección de sus primeras filas, dimensiones y tipos de datos iniciales.

In [3]:
# Carga de datasets
df_estaciones = pd.read_csv('BaseUnificadaEstaciones.csv')
df_tarifas = pd.read_csv('registro-historico-del-precio-del-boleto.csv')
df_viajes = pd.read_csv('viajes_anual.csv')

print(f"Base Estaciones: {df_estaciones.shape[0]:,} registros | {df_estaciones.shape[1]} columnas")
print(f"Histórico Tarifas: {df_tarifas.shape[0]} registros | {df_tarifas.shape[1]} columnas")
print(f"Viajes Anuales: {df_viajes.shape[0]} registros | {df_viajes.shape[1]} columnas")

# Muestreo visual de la base principal
df_estaciones.head()

Base Estaciones: 822,363 registros | 6 columnas
Histórico Tarifas: 304 registros | 4 columnas
Viajes Anuales: 48 registros | 3 columnas


,FECHA,ESTACION,LINEA,DESDE,HORA,CANTIDAD
0,3/1/2020,Acoyte,A,7:45,7,20
1,3/1/2020,Acoyte,A,8:00,8,16
2,3/1/2020,Acoyte,A,8:15,8,22
3,3/1/2020,Acoyte,A,8:30,8,27
4,3/1/2020,Acoyte,A,8:45,8,37


# Auditoría de calidad de datos: nulos, tipologías y anomalías
Se examina la presencia de valores faltantes (NaN), duplicados y la consistencia en los campos de texto categóricos.

In [4]:
# Verificación de valores nulos
print("--- Valores nulos en BaseUnificadaEstaciones ---")
print(df_estaciones.isnull().sum())

# Detección de inconsistencias en la columna LINEA (espacios residuales)
print("\nValores únicos iniciales en LINEA:")
print(repr(df_estaciones['LINEA'].unique().tolist()))

# Resumen estadístico del volumen de validaciones
print("\nEstadísticas del campo CANTIDAD:")
print(df_estaciones['CANTIDAD'].describe())

--- Valores nulos en BaseUnificadaEstaciones ---
FECHA       0
ESTACION    0
LINEA       0
DESDE       0
HORA        0
CANTIDAD    0
dtype: int64

Valores únicos iniciales en LINEA:
['A', 'B', 'C', 'D', 'E', 'H', 'D              ', 'A              ', 'B              ', 'C              ', 'E              ', 'H              ']

Estadísticas del campo CANTIDAD:
count    822363.000000
mean         31.994799
std          76.414429
min           1.000000
25%           5.000000
50%          12.000000
75%          29.000000
max        3827.000000
Name: CANTIDAD, dtype: float64


# Limpieza, transformación y feature engineering (ETL)
En esta fase se subsanan las inconsistencias de texto, se estandarizan las marcas temporales a formato Datetime y se crean variables operativas clave para el análisis de flujos (franjas pico/valle y tipo de día hábil).

In [5]:
# 5.1 Depuración de espacios en blanco en la identificación de Líneas
df_estaciones['LINEA'] = df_estaciones['LINEA'].str.strip()

# 5.2 Parseo estandarizado de fechas y días de la semana
df_estaciones['FECHA'] = pd.to_datetime(df_estaciones['FECHA'])
df_estaciones['DIA_SEMANA'] = df_estaciones['FECHA'].dt.dayofweek # 0=Lunes, 6=Domingo
df_estaciones['ES_HABIL'] = df_estaciones['DIA_SEMANA'] < 5

# 5.3 Clasificación en franjas horarias operativas
def clasificar_franja(hora):
    if hora in [7, 8, 9]:
        return 'Pico Mañana'
    elif hora in [16, 17, 18]:
        return 'Pico Tarde'
    return 'Valle'

df_estaciones['FRANJA'] = df_estaciones['HORA'].apply(clasificar_franja)

print("Transformaciones completadas con éxito:")
print(f"- Líneas unificadas: {df_estaciones['LINEA'].unique().tolist()}")
print(f"- Rango temporal: {df_estaciones['FECHA'].min().date()} al {df_estaciones['FECHA'].max().date()}")
df_estaciones.head(3)

Transformaciones completadas con éxito:
- Líneas unificadas: ['A', 'B', 'C', 'D', 'E', 'H']
- Rango temporal: 2020-03-01 al 2020-11-19


,FECHA,ESTACION,LINEA,DESDE,HORA,CANTIDAD,DIA_SEMANA,ES_HABIL,FRANJA
0,2020-03-01,Acoyte,A,7:45,7,20,6,False,Pico Mañana
1,2020-03-01,Acoyte,A,8:00,8,16,6,False,Pico Mañana
2,2020-03-01,Acoyte,A,8:15,8,22,6,False,Pico Mañana


# Análisis Exploratorio de Datos (EDA) y validación de métricas
Se contrastan los patrones de afluencia: relación hábil vs. fin de semana, concentración en horas pico y estimación de la elasticidad precio de la demanda frente al costo del boleto.

In [6]:
# 6.1 Relación Hábil vs. Fin de Semana
promedio_habil = df_estaciones[df_estaciones['ES_HABIL']].groupby('FECHA')['CANTIDAD'].sum().mean()
promedio_finde = df_estaciones[~df_estaciones['ES_HABIL']].groupby('FECHA')['CANTIDAD'].sum().mean()
ratio_dias = promedio_habil / promedio_finde

print(f"Promedio diario hábil: {promedio_habil:,.0f} pasajeros")
print(f"Promedio diario fin de semana: {promedio_finde:,.0f} pasajeros")
print(f"Razón hábil / no hábil: {ratio_dias:.2f} a 1")

# 6.2 Top 10 de estaciones con mayor afluencia
top_estaciones = df_estaciones.groupby('ESTACION')['CANTIDAD'].sum().nlargest(10).reset_index()
print("\nTop 10 estaciones nodales:")
print(top_estaciones)

# 6.3 Módulo econométrico: Elasticidad de Arco (2018 - 2019)
viajes_anuales = df_viajes.groupby('year')['total'].sum().reset_index()
tarifas_anuales = df_tarifas.groupby('año')['precio'].mean().reset_index()
df_eco = pd.merge(viajes_anuales, tarifas_anuales, left_on='year', right_on='año')

# Cálculo formal de elasticidad arco para el salto tarifario 2018-2019
q0 = df_eco.loc[df_eco['year'] == 2018, 'total'].values[0]
q1 = df_eco.loc[df_eco['year'] == 2019, 'total'].values[0]
p0 = df_eco.loc[df_eco['year'] == 2018, 'precio'].values[0]
p1 = df_eco.loc[df_eco['year'] == 2019, 'precio'].values[0]

elasticidad_arco = ((q1 - q0) / ((q0 + q1) / 2)) / ((p1 - p0) / ((p0 + p1) / 2))
print(f"\nElasticidad de Arco calculada (2018-2019): Ep = {elasticidad_arco:.4f}")

Promedio diario hábil: 124,243 pasajeros
Promedio diario fin de semana: 37,724 pasajeros
Razón hábil / no hábil: 3.29 a 1

Top 10 estaciones nodales:
              ESTACION  CANTIDAD
0         Constitucion   2568335
1               Retiro   1066460
2     Federico Lacroze    910059
3                Rosas    856360
4  Congreso de Tucuman    854368
5          San Pedrito    821131
6    Carlos Pellegrini    696534
7                 Peru    610868
8      Leandro N. Alem    607614
9             Catedral    603787

Elasticidad de Arco calculada (2018-2019): Ep = -0.0388


# Exportación del dataset procesado para el tablero interactivo
Se genera el archivo final enriquecido y limpio. Este archivo es el artefacto que consume de forma directa la aplicación app.py, optimizando los tiempos de respuesta y eliminando redundancias en la visualización.

In [7]:
nombre_salida = 'subte_procesado.csv'
df_estaciones.to_csv(nombre_salida, index=False)

print(f"Archivo generado exitosamente: {nombre_salida}")
print(f"Tamaño optimizado en disco: {os.path.getsize(nombre_salida) / (1024*1024):.2f} MB")
print("El archivo está listo para ser consumido por app.py.")

Archivo generado exitosamente: subte_procesado.csv
Tamaño optimizado en disco: 40.05 MB
El archivo está listo para ser consumido por app.py.
